# Model integration (LangChain 1.3+)

Connect LangChain to **OpenAI**, **Google Gemini**, and **Groq** chat models. Every call returns an `AIMessage` (use `.content` for plain text).

Two ways to construct a model:

| Approach | Example | When to use |
|----------|---------|-------------|
| **`init_chat_model`** | `"gpt-4o-mini"`, `"google_genai:gemini-2.5-flash"`, `"groq:qwen/qwen3-32b"` | One API for many providers |
| **Provider class** | `ChatOpenAI`, `ChatGoogleGenerativeAI`, `ChatGroq` | Extra constructor args (`temperature`, `max_tokens`) |

**Prerequisite:** API keys in repo-root `.env` (`OPENAI_API_KEY`, `GOOGLE_API_KEY`, `GROQ_API_KEY`). Run the env cell first.

**Sections:** 1–3 `init_chat_model` (OpenAI, Gemini, Groq) | 4 provider classes | 5 `invoke` / `stream` / `batch`

In [2]:
"""Load API keys from .env before calling any model.

load_dotenv() puts keys into os.environ. Only re-assign when the value exists
(assigning None raises TypeError).
"""
import os
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

for key in ("OPENAI_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY"):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    else:
        print(f"{key}: missing (add to .env if you need this provider)")

## 1. OpenAI — `init_chat_model`

A bare model name (no prefix) defaults to **OpenAI**. `init_chat_model` returns a `ChatOpenAI` instance that reads `OPENAI_API_KEY` from the environment.

**Flow:** `model.invoke("prompt")` → wraps prompt as `HumanMessage` → API call → `AIMessage`.

In [3]:
# init_chat_model: unified factory — picks provider from the model string.
# No "openai:" prefix needed; bare name defaults to OpenAI.
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4o-mini")  # → ChatOpenAI, reads OPENAI_API_KEY

In [4]:
# Inspect the resolved chat model (type, model_name, client, capabilities in profile)
model

ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x112ef20f0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x112d36ab0>, root_client=<openai.OpenAI object at 0x11208e360>, root_async_client=<openai.AsyncOpenAI object at 0x112817b90>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**

In [5]:
# invoke: str → HumanMessage → API call → AIMessage
# .content = reply text; response_metadata has token usage and model_name
response = model.invoke("How are you?")
response

AIMessage(content="I don't have feelings, but I'm here and ready to help you! How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 11, 'total_tokens': 32, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a9ccf61ec1', 'id': 'chatcmpl-DkpYgIS56efGcsCcYkxoY1vDLir4G', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7377-28ec-7683-bb21-e7e8674721ea-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 21, 'total_tokens': 32, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 2. Google Gemini — `init_chat_model`

Non-OpenAI providers need a **prefix**: `"google_genai:<model-id>"`. Uses `GOOGLE_API_KEY`.

Re-assigning `model` here replaces the OpenAI instance from section 1.

In [6]:
from langchain.chat_models import init_chat_model

# Provider prefix required for non-OpenAI models
model = init_chat_model("google_genai:gemini-2.5-flash")  # → ChatGoogleGenerativeAI

In [7]:
# Inspect: ChatGoogleGenerativeAI, model='gemini-2.5-flash', profile shows capabilities
model

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x111ac7320>, default_metadata=(), model_kwargs={})

In [8]:
# invoke accepts a str (wrapped as HumanMessage) or a list of messages
response = model.invoke("How are you?")
response  # AIMessage — use response.content for text only

AIMessage(content='As an AI, I don\'t have feelings or a physical state, so I can\'t be "how" in the human sense. However, I\'m fully operational and ready to assist you!\n\nHow can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7377-439b-70c0-b266-c3bb113a3b37-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 212, 'total_tokens': 217, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 162}})

## 3. Groq — `init_chat_model`

Format: `"groq:<full-model-id>"`. The id must match Groq docs exactly (e.g. `qwen/qwen3-32b`, **not** `qwen3-32b`). Uses `GROQ_API_KEY`.

**Section 5** reuses this `model` for streaming and batch examples.

In [9]:
# groq: prefix + vendor/model id (must match Groq console docs)
model = init_chat_model("groq:qwen/qwen3-32b")  # → ChatGroq

In [10]:
# Inspect: ChatGroq, model_name='qwen/qwen3-32b'
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x110bea0f0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1141e6390>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [11]:
response = model.invoke("How are you?")
# Qwen on Groq may include <think> reasoning tokens before the reply
response

AIMessage(content='<think>\nLet me think about this in a structured way. The user has asked "How are you?" which is a common social greeting. I need to respond in a way that is both honest and appropriate for an AI.\n\nFirst, since I\'m an AI, I don\'t experience emotions in the biological sense. I should be clear about that. But I can still be friendly and helpful.\n\nSecond, I should acknowledge the sentiment behind the question. People often ask how I\'m doing as a way of being polite and starting conversation.\n\nThird, I should respond in a way that invites further interaction. Maybe offer help or ask how they\'re doing.\n\nFourth, I should keep the tone warm and approachable. Use natural language with some personality.\n\nLet me structure my response:\n1. Acknowledge I\'m an AI and don\'t have feelings\n2. Express that I\'m functioning well and ready to help\n3. Ask how they\'re doing and offer assistance\n4. Use friendly emoji to add warmth\n\nThis should cover the bases - be ho

## 4. Provider-specific classes (alternative to `init_chat_model`)

Import the vendor class directly when you need explicit control:

- `ChatOpenAI`, `ChatGoogleGenerativeAI`, `ChatGroq`
- Pass kwargs: `temperature=0.7`, `max_tokens=500`, etc.
- Same `invoke` / `stream` / `batch` methods as `init_chat_model` models

In [12]:
"""OpenAI — direct ChatOpenAI (equivalent to init_chat_model('gpt-4o-mini'))."""
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")  # OPENAI_API_KEY from env
response = model.invoke("How are you?")
response

AIMessage(content="I'm just a computer program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 11, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a9ccf61ec1', 'id': 'chatcmpl-DkpYwBKX4Yuvm8XZAh4s2x5zSKC8Y', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e7377-671f-7592-9aec-60dbaea55510-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 28, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
"""Google Gemini — direct ChatGoogleGenerativeAI."""
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")  # GOOGLE_API_KEY from env
response = model.invoke("How are you?")
response

AIMessage(content="As an AI, I don't have feelings or personal well-being in the way humans do, but I am functioning perfectly and ready to assist you!\n\nHow are you doing today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7377-7342-7e50-a5ed-6bb3a4c05945-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 5, 'output_tokens': 182, 'total_tokens': 187, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 143}})

In [14]:
"""Groq — direct ChatGroq. Use full model id with vendor prefix."""
from langchain_groq import ChatGroq

# Wrong: "qwen3-32b" → 404 model_not_found
# Right: "qwen/qwen3-32b" (same as init_chat_model("groq:qwen/qwen3-32b"))
model = ChatGroq(model="qwen/qwen3-32b")  # GROQ_API_KEY from env
response = model.invoke("How are you?")
response

AIMessage(content="<think>\nThe user is asking about my condition. As an AI model, I don't have emotions or feelings, but I can interact and answer questions. I need to respond in a friendly and helpful manner. I should acknowledge the user's greeting, mention that I'm an AI with no emotions, but ready to help. Keep the tone warm and approachable to encourage further interaction. Maybe add an emoji to keep it friendly. Avoid overcomplicating the response. Make sure the user knows I'm here to assist them. Alright, let me put that together.\n</think>\n\nHi there! I'm an AI, so I don't experience feelings or emotions, but I'm here and ready to help you with whatever you need! 😊 What can I assist you with today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 12, 'total_tokens': 170, 'completion_time': 0.408744142, 'completion_tokens_details': None, 'prompt_time': 0.000450927, 'prompt_tokens_details': None, 'queue_time': 0.051689558, 't

### Quick reference

| Provider | `init_chat_model` | Direct class | Env var |
|----------|-------------------|--------------|---------|
| OpenAI | `"gpt-4o-mini"` | `ChatOpenAI(model="...")` | `OPENAI_API_KEY` |
| Google | `"google_genai:gemini-2.5-flash"` | `ChatGoogleGenerativeAI(model="...")` | `GOOGLE_API_KEY` |
| Groq | `"groq:qwen/qwen3-32b"` | `ChatGroq(model="qwen/qwen3-32b")` | `GROQ_API_KEY` |

| Method | Returns |
|--------|---------|
| `model.invoke(prompt)` | One `AIMessage` |
| `model.stream(prompt)` | Iterator of chunks |
| `model.batch([...])` | `list[AIMessage]` |

**Next:** Section 5 — streaming, batch, and batch options.

## 5. Streaming and batch

| Method | Returns | When to use |
|--------|---------|-------------|
| `model.invoke(prompt)` | One `AIMessage` (full reply at once) | Simple Q&A, short answers |
| `model.stream(prompt)` | Iterator of chunks | Chat UIs, long text, tokens as they arrive |
| `model.batch([...])` | List of `AIMessage` | Many independent prompts in one call |

**Prerequisite:** Run section 3 or 4 first so `model` is a Groq `ChatGroq` instance (examples below use Groq). All three methods work with OpenAI and Gemini too.

**Subsections:** 5a streaming | 5b batch | 5c `max_concurrency` & `max_tokens`

### 5a. Streaming (`stream`)

Models generate tokens incrementally. `stream()` returns a **generator** of chunks (often `AIMessageChunk`). Loop and print each chunk for a typewriter effect.

- `chunk.text` or `chunk.content` — partial text in this chunk
- `end=""`, `flush=True` — print on one line without extra newlines
- Reasoning models (e.g. Qwen on Groq) may stream `<think>` before the final answer

In [19]:
# stream() alone returns a generator — iterate in the next cell to see tokens
# Uses `model` from Groq section (section 3 or 4)
model.stream("write a short story about a cat in 25 words")

<generator object BaseChatModel.stream at 0x1140d1d50>

In [25]:
# Consume the stream: each chunk is a small piece of the full response
for chunk in model.stream("write a short story about a cat in 25 words"):
    print(chunk.text, end="", flush=True)  # .content also works on many models

<think>
Okay, the user wants a short story about a cat in exactly 25 words. Let me start by thinking about the key elements. A cat as the main character, maybe some action or adventure. Need to keep it concise.

First, establish the setting. Maybe a small town or a specific location. A cat named Whiskers? Classic but recognizable. What's the conflict or event? Maybe something magical to make it interesting. A glowing key? That adds a touch of mystery.

Next, the cat finding the key. Where would it find it? Behind the bakery? That's a specific detail. Then, the key leads to a door. The door could open to a garden with stars. That's imaginative and gives a magical feel. 

Now, count the words. Let me draft a sentence: "Whiskers found a glowing key behind the bakery. It unlocked a door to a garden where stars bloomed. The moon smiled." Let me count: 1-Whiskers, 2-found, 3-a, 4-glowing, 5-key, 6-behind, 7-the, 8-bakery. 9-It, 10-unlocked, 11-a, 12-door, 13-to, 14-a, 15-garden, 16-where, 17

### `stream` vs `invoke`

| | `invoke` | `stream` |
|--|----------|----------|
| Returns | One complete `AIMessage` | Generator of `AIMessageChunk` |
| Waits until | Entire reply is done | First token arrives |
| Best for | Short answers, simple scripts | Chat UIs, long generations |


In [ ]:
# Same prompt as streaming above — entire reply returned at once
response = model.invoke("write a short story about a cat in 25 words")
response.content

### 5b. Batch (`batch`)

Send **multiple independent prompts** in one call. LangChain can run them **in parallel**, which is faster than a loop of `invoke` calls.

- **Input:** `list[str]` — each string is a separate one-shot prompt (not multi-turn chat)
- **Output:** `list[AIMessage]` in the **same order** as the input list
- **Inspect:** return the list directly, or loop and use `.content` for readable text

In [ ]:
# Returns list[AIMessage] — one per prompt
responses = model.batch([
    "What is the capital of France?",
    "What is the capital of Germany?",
    "What is the capital of Italy?",
])
responses

In [ ]:
# Same batch — print only the reply text from each AIMessage
for response in model.batch([
    "What is the capital of France?",
    "What is the capital of Germany?",
    "What is the capital of Italy?",
]):
    print(response.content)
    print("-" * 40)

### 5c. Batch options: `max_concurrency` and `max_tokens`

| Option | How to pass | Effect |
|--------|-------------|--------|
| `max_concurrency` | `batch(..., config={"max_concurrency": N})` | Limit parallel API calls (avoid rate limits) |
| `max_tokens` | `batch(..., max_tokens=N)` | Cap completion length per response |

Model kwargs (`temperature`, `max_tokens`, etc.) can be passed to `invoke`, `stream`, and `batch` the same way.

In [ ]:
# Run at most 2 requests in parallel
model.batch(
    ["What is the capital of France?", "What is the capital of Germany?"],
    config={"max_concurrency": 2},
)

In [ ]:
# Stop each completion after 100 tokens (finish_reason may be 'length')
# Reasoning models (Qwen) may cut off mid-thought when limit is low
model.batch(
    ["What is the capital of France?", "What is the capital of Germany?"],
    max_tokens=100,
)

### Section 5 summary

```python
model.invoke("one prompt")                        # -> AIMessage
model.stream("one prompt")                        # -> iterator of chunks
model.batch(["a", "b", "c"])                      # -> [AIMessage, ...]
model.batch([...], config={"max_concurrency": 2}) # limit parallelism
model.batch([...], max_tokens=100)                # cap output length
```

**Async:** `ainvoke`, `astream`, `abatch` — same signatures, use `await` or `async for`.